In [176]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from decorator import decorate

torch.manual_seed(12046)

In [177]:
sequence_len = 64
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("使用MPS！")
else:
    device = torch.device("cpu")
    print("使用CPU")

使用MPS！


In [178]:
# from RNN note
from torch import nn
import torch.nn.functional as F
# 检查是否支持MPS
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("使用MPS！")
else:
    device = torch.device("cpu")
    print("使用CPU")
import torch.optim as optim
from torch.utils.data import DataLoader
from datasets import load_dataset
import matplotlib.pyplot as plt

# 使用通配符匹配所有分片
data_path = "/Users/hujinjia/PycharmProjects/JupyterProject/python/python/final/jsonl/train/*.jsonl.gz"
datasets = load_dataset('json', data_files=data_path)
datasets = datasets['train'].filter(lambda x: 'apache/spark' in x['repo'])
print(datasets[8]['original_string'])

class CharTokenizer:
    def __init__(self, data, end_ind=0):
        chars = sorted(list(set(''.join(data))))
        self.char2ind = {s: i + 1 for i, s in
                         enumerate(chars)}  # 创建 {字符: 索引} 的映射。self.char2ind = {'a': 2, 'b': 3, 'c': 4}
        self.char2ind['<|e|>'] = end_ind
        self.ind2char = {v: k for k, v in
                         self.char2ind.items()}  # 创建从索引到字符的反向映射，便于解码。self.ind2chat = {0: '<|b|>', 1: '<|e|>', 2: 'a', 3: 'b', 4: 'c'}
        self.end_ind = end_ind

    def encode(self, x):
        return [self.char2ind[i] for i in x]

    def decode(self, x):
        if isinstance(x, int):
            return self.ind2char[x]
        else:
            return [self.ind2char[i] for i in x]


tokenizer = CharTokenizer(datasets['original_string'])
#测试
test_str = 'def f(x)'
re = tokenizer.encode(test_str)
print(re)
''.join(tokenizer.decode(range(len(tokenizer.char2ind))))

使用MPS！
def to_arrow_schema(schema):
    """ Convert a schema from Spark to Arrow
    """
    import pyarrow as pa
    fields = [pa.field(field.name, to_arrow_type(field.dataType), nullable=field.nullable)
              for field in schema]
    return pa.schema(fields)
[70, 71, 72, 2, 72, 10, 90, 11]


'<|e|>\n !"#$%&\'()*+,-./0123456789:;<=>?@ABCDEFGHIJKLMNOPQRSTUVWXYZ[\\]^_`abcdefghijklmnopqrstuvwxyz{|}~ö'

In [179]:
# k:(B,T,H) H:特征个数
# Q:(B,T,H)
# k @ Q.transpose(-2,-1) : (B,T,T)

In [180]:
# 创造半三角的例子
scores =  torch.randn(1,4,4)
print(scores)

tensor([[[ 1.0185, -1.3091,  1.2908,  0.5276],
         [-0.2985,  1.6259,  2.0433, -0.6417],
         [ 0.8795, -1.0512,  1.1491,  0.6116],
         [ 0.2128, -0.5512,  0.0450,  0.5010]]])


In [181]:
# 定义下三角矩阵
tril = torch.tril(torch.ones(4,4))
print(tril)

tensor([[1., 0., 0., 0.],
        [1., 1., 0., 0.],
        [1., 1., 1., 0.],
        [1., 1., 1., 1.]])


In [182]:
s = scores.masked_fill(tril ==0, float('-inf'))
print(s) #行的和不一定为 1，代表的是相对重要性的原始度量

tensor([[[ 1.0185,    -inf,    -inf,    -inf],
         [-0.2985,  1.6259,    -inf,    -inf],
         [ 0.8795, -1.0512,  1.1491,    -inf],
         [ 0.2128, -0.5512,  0.0450,  0.5010]]])


In [183]:
# 定义权重分布
F.softmax(s, dim= -1) #每一行的和为 1, 代表的是归一化后的注意力权重

tensor([[[1.0000, 0.0000, 0.0000, 0.0000],
         [0.1274, 0.8726, 0.0000, 0.0000],
         [0.4074, 0.0591, 0.5335, 0.0000],
         [0.2743, 0.1278, 0.2319, 0.3659]]])

In [184]:
# softmax对方差的敏感性
x = torch.randn(1,8)
x.var()

tensor(1.1908)

In [185]:
print(F.softmax(x, dim= -1))

tensor([[0.1236, 0.0400, 0.0791, 0.0175, 0.6045, 0.0262, 0.0633, 0.0457]])


In [186]:
# Softmax 的"锐化"效应，当输入值被放大时，Softmax 的输出分布会变得更加"极端"。
print(F.softmax(x*100, dim= -1))

tensor([[0., 0., 0., 0., 1., 0., 0., 0.]])


In [187]:
# 对齐分数的方差变化
B, T, H = 32, 100, 10
K = torch.randn(B, T, H)
Q = torch.randn(B, T, H)
scores = K @ Q.transpose(-2,-1)/H ** 0.5
scores.var() #保持方差为1避免， Softmax 进入饱和区（梯度消失），保持梯度稳定性， 使训练更稳定

tensor(1.0130)

In [188]:
def attention(query, key, value, dropout, mask=None):
    # query:             (B, T, H)
    # mask:                 (T, T)
    # output:            (B, T, H)
    B, T, H = query.shape
    scores = query @ key.transpose(-2,-1) / H ** 0.5
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    w_att = F.softmax(scores, dim=-1)  # (B, T, T)
    out = w_att @ value                # (B, T, H)
    return out

In [189]:
#实现单向自注意力
class MaskedAttention(nn.Module):
    # emb_size:输入向量长度，head_size：背景向量长度
    """
    RNN：需要偏置来拟合数据中的偏移
    Transformer：偏置会被注意力分数计算"抵消"
    """
    def __init__(self, emb_size, head_size):
        # emb_size: C, head_size: H
        super(MaskedAttention, self).__init__()
        self.key = nn.Linear(emb_size, head_size, bias=False)
        self.query = nn.Linear(emb_size, head_size, bias=False)
        self.value = nn.Linear(emb_size, head_size, bias=False)
        # 定义下三角
        self.register_buffer('trill', torch.tril(torch.ones(sequence_len,sequence_len))) #注册为缓冲区，使mask的执行设备一致
        self.dp = nn.Dropout(0.4)

    def forward(self, x):
        #  x : (B, T, C)
        # out: (B, T, C)
        B, T, C = x.shape
        k = self.key(x)     # (B, T, H)
        q = self.query(x)   # (B, T, H)
        v = self.value(x)   # (B, T, H)
        mask = self.trill[:T, :T]
        out = attention(q, k, v, self.dp, mask) #使用注册的缓冲区
        return out

In [190]:
# 参数
emb_size = 128
head_size = 8
n_layer = 12
sequence_len = 64 # 模型可以处理的最大文本长度
learning_rate = 0.003
eval_iters = 20
batch_size = 500

In [191]:
class MaskedMultiHeadAttention(nn.Module):
    def __init__(self, emb_size, head_size):
        super(MaskedMultiHeadAttention, self).__init__()
        # 计算单头注意力的个数
        n_head = emb_size // head_size
        heads = [MaskedAttention(emb_size, head_size) for _ in range(n_head)] #
        self.heads = nn.ModuleList(heads) #确保列表中的所有子模块的参数被正确注册到主模块中
        self.proj = nn.Linear(emb_size, emb_size)
        self.dp = nn.Dropout(0.4)

    def forward(self, x):
        # x:  (B, T, C)
        #out  (B, T, C)
        out = torch.concat([h(x) for h in self.heads], dim=-1) #(B, T, C)
        out = self.dp(self.proj(out))                         #(B, T, C)
        return out

In [192]:
class FeedForward(nn.Module):
    def __init__(self, emb_size):
        super(FeedForward, self).__init__()
        self.ln1 = nn.Linear(emb_size, 4 * emb_size)
        self.ln2 = nn.Linear(4 * emb_size, emb_size)
        self.dp = nn.Dropout(0.4)

    def forward(self, x):
        # x: (B, T, C)
        out = F.gelu(self.ln1(x))  #(B, T, C)
        out = self.dp(self.ln2(out)) #(B, T, C)
        return out

In [193]:
# 构造解码块
class Block(nn.Module):
    def __init__(self, emb_size, head_size):
        super(Block, self).__init__()
        self.l1 = nn.LayerNorm(emb_size)
        self.mha = MaskedMultiHeadAttention(emb_size, head_size)
        self.l2 = nn.LayerNorm(emb_size)
        self.ff = FeedForward(emb_size)

    def forward(self, x):
        """
        残差连接
        """
        # x: (B, T, C)
        #0ut:(B, T, C)
        x = x + self.mha(self.l1(x))
        x = x + self.ff(self.l2(x))
        return x

In [194]:
class CharGPT(nn.Module):

    def __init__(self, vs):
        super(CharGPT, self).__init__()
        self.token_emb = nn.Embedding(vs, emb_size) #捕捉语义信息
        self.pos_emb = nn.Embedding(sequence_len, emb_size) #捕捉位置信息
        block = [Block(emb_size, head_size) for _ in range(n_layer)]
        self.block = nn.Sequential(*block) # * 号把列表解包成多个参数
        self.l = nn.LayerNorm(emb_size)
        self.lm = nn.Linear(emb_size, vs)

    def forward(self,x):
        # x:[B, T]
        # logits: (B, T, vs)
        B, T = x.shape
        pos = torch.arange(0, T, device=device) #创建位置索引序列
        token_embbings = self.token_emb(x)  #(B, T, C)
        pos_embbings = self.pos_emb(pos)    #(B, T, C)
        # 实验证明，在足够的维度空间中，加法足以让模型区分和利用这两种信息。
        h = token_embbings + pos_embbings   #(B, T, C)
        h = self.block(h)                   #(B, T, C)
        logits = self.lm(self.l(h))         #(B, T, vs)
        return logits

In [195]:
c_model = CharGPT(len(tokenizer.char2ind)).to(device)
c_model

CharGPT(
  (token_emb): Embedding(98, 128)
  (pos_emb): Embedding(64, 128)
  (block): Sequential(
    (0): Block(
      (l1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (mha): MaskedMultiHeadAttention(
        (heads): ModuleList(
          (0-15): 16 x MaskedAttention(
            (key): Linear(in_features=128, out_features=8, bias=False)
            (query): Linear(in_features=128, out_features=8, bias=False)
            (value): Linear(in_features=128, out_features=8, bias=False)
            (dp): Dropout(p=0.4, inplace=False)
          )
        )
        (proj): Linear(in_features=128, out_features=128, bias=True)
        (dp): Dropout(p=0.4, inplace=False)
      )
      (l2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (ff): FeedForward(
        (ln1): Linear(in_features=128, out_features=512, bias=True)
        (ln2): Linear(in_features=512, out_features=128, bias=True)
        (dp): Dropout(p=0.4, inplace=False)
      )
    )
    (1): Block(
   

In [196]:
sum(p.numel() for p in c_model.parameters())

2408290

In [199]:
@torch.no_grad()
# 生成字
def generate(model, context,tokenizer,max_new_tokens=300):
    # context:(B,T), B =1
    # out = []
    out = context.tolist()[0] #将背景放到输入中
    model.eval()
    for _ in range(max_new_tokens):
        logits = model(context[:, -sequence_len:]) #(B,T,VS) -> (1,T,98)
        probs = F.softmax(logits[:,-1,:], dim=-1) # 取最后一个logits (1, 98)
        #随机生成文本
        ix = torch.multinomial(probs, num_samples=1)  #(1,1)
        # 更新背景
        context = torch.concat((context, ix), dim=-1)
        out.append(ix.item())
        if out[-1] == tokenizer.end_ind:
            break
    model.train()
    return out

In [201]:
# 检查上面，生成未训练的文本
context = torch.zeros((1,10), dtype=torch.long, device = device)
print(''.join(tokenizer.decode(generate(c_model, context,tokenizer))))

<|e|><|e|><|e|><|e|><|e|><|e|><|e|><|e|><|e|><|e|>
:?e,$OCMXNdUB
K#GL7z>"s7sj9N$-[kdl
ZU#RnX7[+jr&|"r:izp8OE1RFV*]<-8eS1R9[>(3udS=Cq'7>m3=d[R'7TR_C+[3n-|4}L|\O[d6z#:;Kg3- fXRcgcBuöWUu6F
I`ö]9^G|o9jwL+]jWöT"(}wVGHö+iPQD!n2SK3C(Ju1a(XW+XFYSlb_mms(jvqWTF=m>di+1\FFoBgK* ,g5VDhhc<|e|>


In [203]:
def process(data, tokenizer, sequence_len):
    text = data['original_string']
    inputs, labels = [],[]
    for t in text:
        enc = tokenizer.encode(t)
        enc += [tokenizer.end_ind]
        for i in range(len(enc) - sequence_len):
            inputs.append(enc[i:i+sequence_len])
            labels.append(enc[i+1:i+1+sequence_len])
    return{'inputs': inputs, 'labels': labels}

In [204]:
tokenized = datasets.train_test_split(test_size = 0.1, seed = 1024, shuffle = True)
f = lambda x: process(x, tokenizer,sequence_len = 64)
tokenized = tokenized.map(f, batched=True, remove_columns = datasets.column_names)
tokenized.set_format(type='torch', device = device)

In [205]:
train_loader = DataLoader(tokenized['train'], batch_size=1000, shuffle=True)
test_loader = DataLoader(tokenized['test'], batch_size=1000, shuffle=True)

In [209]:
def estimate_loss(model):
    re = {} #创建一个空字典，用于存储训练损失和测试损失,最终返回的结构：{'train': 0.123, 'test': 0.456}
    model.eval() #将训练模式切换为评估模式
    train_loss = _loss(model, train_loader)
    test_loss = _loss(model, test_loader)
    re['train'] = train_loss
    re['test'] = test_loss
    model.train()  #切换回训练模式
    return re

@torch.no_grad()
def _loss(model, data_loader):
    #计算模型在不同数据集下面的评估指标
    loss = []
    data_iter = iter(data_loader)
    # 随机使用多个批量数据来评估模型效果
    for k in range(eval_iters):
        data = next(data_iter, None) #如果没有数据，返回 None
        if data is None: # 如果当前迭代器没有数据了
            data_iter = iter(data_loader) # 重新创建迭代器
            data = next(data_iter, None) # 获取第一个 batch
        inputs, labels = data['inputs'], data['labels'] # (B, T)
        logits = model(inputs)                          # (B, T, vs)
        # 官网交叉熵需要的纬度（B ，VS ，...)
        loss.append(F.cross_entropy(logits.transpose(-2,-1), labels).item()) # item() 将损失值(张量）转换为 Python 数字
    return torch.tensor(loss).mean().item()

estimate_loss(c_model)

{'train': 3.303755283355713, 'test': 3.336411952972412}

In [207]:
def train_model(model,optimizer,epochs=10):
    lossi = [] # 记录模型在训练集上的模型损失
    for epoch in range(epochs):
        for i, data in enumerate(train_loader, 0):
            inputs, labels = data['inputs'], data['labels'] # (B, T)
            optimizer.zero_grad()
            logits = model(inputs)                          # (B ,T vs)
            loss = F.cross_entropy(logits.transpose(-2,-1), labels)          # 如上
            lossi.append(loss.item())
            loss.backward()
            optimizer.step()
        # 评估模型，并输出结果
        stats = estimate_loss(model)
        train_loss = f'train loss {stats['train']:.4f}'
        test_loss = f'test loss {stats['test']:.4f}'
        print(f'epoch {epoch:>2}: {train_loss}, {test_loss}')
    return lossi

In [210]:
l = train_model(c_model, optim.Adam(c_model.parameters(), lr=learning_rate))

RuntimeError: MPS backend out of memory (MPS allocated: 18.10 GiB, other allocations: 34.92 MiB, max allowed: 18.13 GiB). Tried to allocate 1.95 MiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).